# What Is the Best Way to Reduce Fatigue in Long COVID?

Analysis of treatment sentiment reports from Long COVID communities (1-month snapshot, `polina_onemonth.db`).  
We identify users who mention **fatigue** in any post, then examine which treatments they discuss, how positively they report on them, and whether fatigue users respond differently from the general population.

**Method:** User-level aggregation (one data point per user per drug), binomial tests against a 50% positive baseline, forest plots with 95% CIs, and Mann-Whitney U comparisons between fatigue and non-fatigue cohorts.

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy import stats

DB_PATH = r"C:\Users\scgee\OneDrive\Documents\Projects\PatientPunk\polina_onemonth.db"
conn = sqlite3.connect(DB_PATH)

# ---------- Sentiment score mapping ----------
SENTIMENT_SCORES = {
    "positive": 1.0,
    "mixed":    0.5,
    "neutral":  0.0,
    "negative": -1.0,
}

# ---------- Outcome classification thresholds ----------
def classify_outcome(avg_score):
    if avg_score > 0.7:
        return "positive"
    elif avg_score < -0.3:
        return "negative"
    else:
        return "mixed/neutral"

# ---------- Generic treatment filter ----------
# These are categories, not actionable treatments a patient can try.
GENERIC_TREATMENTS = {
    "supplements", "medication", "treatment", "therapy", "vitamin",
}

def is_generic(name):
    """Return True if the treatment name is a generic category word."""
    return name.strip().lower() in GENERIC_TREATMENTS

plt.rcParams["figure.figsize"] = (10, 6)
print(f"Connected to {DB_PATH}")

## 2. Data Exploration -- The Fatigue Cohort

The **fatigue cohort** comprises all users who mention "fatigue" in the body or title of any post. This captures the broadest set of users self-reporting this symptom.

In [ ]:
# Define fatigue cohort
fatigue_users = pd.read_sql("""
    SELECT DISTINCT user_id
    FROM posts
    WHERE LOWER(body_text) LIKE '%fatigue%'
       OR LOWER(title) LIKE '%fatigue%'
""", conn)

fatigue_user_ids = set(fatigue_users["user_id"])
print(f"Fatigue cohort: {len(fatigue_user_ids)} users")

# Pull all treatment reports for fatigue users
fatigue_reports = pd.read_sql("""
    SELECT tr.report_id, tr.user_id, tr.drug_id, tr.sentiment,
           t.canonical_name AS drug
    FROM treatment_reports tr
    JOIN treatment t ON t.id = tr.drug_id
    WHERE tr.user_id IN (
        SELECT DISTINCT user_id FROM posts
        WHERE LOWER(body_text) LIKE '%fatigue%'
           OR LOWER(title) LIKE '%fatigue%'
    )
""", conn)

# Map sentiment to numeric scores
fatigue_reports["score"] = fatigue_reports["sentiment"].map(SENTIMENT_SCORES)

n_reporters = fatigue_reports["user_id"].nunique()
n_reports   = len(fatigue_reports)
n_drugs     = fatigue_reports["drug"].nunique()

print(f"Fatigue users with treatment reports: {n_reporters}")
print(f"Total treatment reports from fatigue users: {n_reports}")
print(f"Distinct treatments mentioned: {n_drugs}")
print(f"\nSentiment distribution:")
print(fatigue_reports["sentiment"].value_counts())

In [ ]:
# ---------- User-level aggregation ----------
# One data point per user per drug (avoids inflating n with repeat posts)
user_drug = (
    fatigue_reports
    .groupby(["user_id", "drug"])
    .agg(avg_score=("score", "mean"), n_posts=("report_id", "count"))
    .reset_index()
)
user_drug["outcome"] = user_drug["avg_score"].apply(classify_outcome)

print(f"User-drug pairs (one data point per user per drug): {len(user_drug)}")
print(f"\nOutcome distribution across all user-drug pairs:")
print(user_drug["outcome"].value_counts())

## 3. Question 1 -- Top Treatments Among Fatigue Users

For each treatment with at least 5 unique fatigue reporters, we compute the user-level positive outcome rate, then run a **one-sample binomial test** (H0: positive rate = 50%) to identify treatments with positive rates that exceed chance.

Generic category words ("supplements", "medication", "treatment", "therapy", "vitamin") are filtered out -- these are not specific treatments a patient can act on.

In [ ]:
# ---------- Per-drug summary ----------
drug_summary = (
    user_drug
    .groupby("drug")
    .agg(
        n_users=("user_id", "nunique"),
        mean_score=("avg_score", "mean"),
        std_score=("avg_score", "std"),
    )
    .reset_index()
)

# Outcome counts per drug
outcome_counts = (
    user_drug
    .groupby(["drug", "outcome"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
for col in ["positive", "mixed/neutral", "negative"]:
    if col not in outcome_counts.columns:
        outcome_counts[col] = 0

drug_summary = drug_summary.merge(outcome_counts, on="drug")

# Filter generics and require >= 5 users
drug_summary = drug_summary[~drug_summary["drug"].apply(is_generic)].copy()
drug_summary = drug_summary[drug_summary["n_users"] >= 5].copy()

# Compute percentages
drug_summary["pct_positive"] = (drug_summary["positive"] / drug_summary["n_users"] * 100).round(1)
drug_summary["pct_negative"] = (drug_summary["negative"] / drug_summary["n_users"] * 100).round(1)
drug_summary["pct_mixed"]    = (drug_summary["mixed/neutral"] / drug_summary["n_users"] * 100).round(1)

# ---------- Binomial test: positive rate > 50%? ----------
binom_results = []
for _, row in drug_summary.iterrows():
    n = int(row["n_users"])
    k = int(row["positive"])
    result = stats.binomtest(k, n, p=0.5, alternative="greater")
    ci = result.proportion_ci(confidence_level=0.95, method="wilson")
    binom_results.append({
        "drug": row["drug"],
        "binom_p": result.pvalue,
        "ci_low": round(ci.low * 100, 1),
        "ci_high": round(ci.high * 100, 1),
        "significant": result.pvalue < 0.05,
    })

binom_df = pd.DataFrame(binom_results)
drug_summary = drug_summary.merge(binom_df, on="drug")
drug_summary = drug_summary.sort_values("n_users", ascending=False)

# ---------- Display top 20 ----------
display_cols = ["drug", "n_users", "mean_score", "pct_positive", "pct_mixed",
                "pct_negative", "ci_low", "ci_high", "binom_p", "significant"]
display_df = drug_summary[display_cols].copy()
display_df.columns = ["Treatment", "N users", "Mean score", "% Positive",
                       "% Mixed/Neutral", "% Negative", "CI low %", "CI high %",
                       "Binom p-value", "Sig (p<.05)"]
display_df["Mean score"] = display_df["Mean score"].round(3)
display_df["Binom p-value"] = display_df["Binom p-value"].apply(
    lambda x: f"{x:.3e}" if x < 0.001 else f"{x:.3f}"
)
display(display_df.head(20))

**How to read this table:**

- **N users** = number of unique fatigue-reporting users who discussed this treatment (one data point per user, regardless of how many posts they made about it).
- **% Positive** = share of those users whose average sentiment was above 0.7 (our "positive" threshold). The **CI low / CI high** columns give the 95% Wilson confidence interval for the true positive rate.
- **Binom p-value** = probability of seeing this many (or more) positive outcomes if the true positive rate were only 50%. Small p-values mean the positive rate is unlikely to be due to chance.

**What this means:** Low dose naltrexone (LDN) has by far the most reporters (79 fatigue users) and the binomial test is highly significant, meaning its positive rate is reliably above 50%. Antihistamines, vitamin D, nicotine, and CoQ10 also have substantial sample sizes. Treatments at the bottom of the table with small n may look impressive (100% positive) but their confidence intervals are wide -- we simply do not have enough data to be sure.

In [ ]:
# ---------- Diverging bar chart: top 15 treatments by user count ----------
top15 = drug_summary.sort_values("n_users", ascending=False).head(15)
top15 = top15.sort_values("n_users", ascending=True).copy()  # flip for horizontal bar

fig, ax = plt.subplots(figsize=(10, 7))

y = np.arange(len(top15))
pct_pos = top15["pct_positive"].values
pct_mix = top15["pct_mixed"].values
pct_neg = top15["pct_negative"].values

# Right side: positive bars extend rightward from 0
ax.barh(y, pct_pos, left=0, color="#2ecc71", label="Positive",
        edgecolor="white", linewidth=0.5)

# Left side: mixed INNERMOST (adjacent to center 0%), negative OUTERMOST
# Step 1: plot mixed from 0 leftward
ax.barh(y, -pct_mix, left=0, color="#bdc3c7", label="Mixed / Neutral",
        edgecolor="white", linewidth=0.5)
# Step 2: plot negative from edge of mixed, extending further left
ax.barh(y, -pct_neg, left=-pct_mix, color="#e74c3c", label="Negative",
        edgecolor="white", linewidth=0.5)

# n-labels on right edge
for i, (_, row) in enumerate(top15.iterrows()):
    ax.text(pct_pos[i] + 1.5, i, f'n={int(row["n_users"])}',
            va="center", ha="left", fontsize=8, color="#555555")

ax.set_yticks(y)
ax.set_yticklabels([d.title() for d in top15["drug"]])
ax.axvline(0, color="black", linewidth=0.8)
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{abs(x):.0f}%"))
ax.set_xlabel("\u2190 Negative / Mixed outcome rate  |  Positive outcome rate \u2192")
ax.set_title("Top 15 Treatments Among Long COVID Fatigue Users\n"
             "(Diverging bar: outcome distribution, sorted by sample size)",
             pad=12)

ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.08),
    ncol=3,
    fontsize=9,
    frameon=False,
)

left_max = max(pct_mix + pct_neg)
ax.set_xlim(-left_max - 10, max(pct_pos) + 15)
plt.tight_layout()
plt.show()

**What this chart shows:** Each horizontal bar represents one treatment. Green bars extend rightward showing the percentage of fatigue users who had a positive outcome. On the left, grey bars (mixed/neutral) sit closest to the center line and red bars (negative) extend further outward. Treatments are sorted from top (fewest reporters) to bottom (most reporters). The `n=` label shows how many unique fatigue users reported on each treatment.

**What this means:** Low dose naltrexone dominates in both sample size (n=79) and positive rate. CoQ10, magnesium, and vitamin D also show strong positive signals. SSRIs stand out as the only top-15 treatment with a net negative leaning among fatigue users, which aligns with community reports of worsening post-exertional symptoms. Antihistamines are widely discussed but show more mixed results, suggesting variable individual responses.

## 4. Question 2 -- Forest Plot: Treatment Sentiment with 95% Confidence Intervals

A forest plot showing the mean user-level sentiment score and its 95% confidence interval for each of the top 15 treatments (by sample size) among fatigue users. This visualizes both the central estimate and the precision of that estimate.

In [ ]:
# ---------- Forest plot: mean sentiment +/- 95% CI ----------
top15_drugs = (
    drug_summary.sort_values("n_users", ascending=False)
    .head(15)["drug"]
    .tolist()
)

forest_data = []
for drug in top15_drugs:
    scores = user_drug.loc[user_drug["drug"] == drug, "avg_score"]
    n = len(scores)
    mean = scores.mean()
    se = scores.std(ddof=1) / np.sqrt(n)
    t_crit = stats.t.ppf(0.975, df=n - 1)
    ci_low = mean - t_crit * se
    ci_high = mean + t_crit * se
    forest_data.append({
        "drug": drug,
        "n": n,
        "mean": mean,
        "ci_low": ci_low,
        "ci_high": ci_high,
    })

forest_df = pd.DataFrame(forest_data).sort_values("mean", ascending=True)

fig, ax = plt.subplots(figsize=(9, 7))

y_pos = np.arange(len(forest_df))

# CI lines
ax.hlines(y_pos, forest_df["ci_low"], forest_df["ci_high"],
          color="#2c3e50", linewidth=1.5, zorder=2)

# Mean dots
ax.scatter(forest_df["mean"], y_pos, color="#2980b9", s=60, zorder=3,
           edgecolors="white", linewidth=0.5)

# Reference lines
ax.axvline(0, color="grey", linewidth=0.8, linestyle="--", alpha=0.6,
           label="Neutral (0.0)")
ax.axvline(0.7, color="#27ae60", linewidth=0.8, linestyle=":", alpha=0.6,
           label="Positive threshold (0.7)")
ax.axvline(-0.3, color="#c0392b", linewidth=0.8, linestyle=":", alpha=0.6,
           label="Negative threshold (-0.3)")

# n-labels
for i, (_, row) in enumerate(forest_df.iterrows()):
    ax.text(max(row["ci_high"], row["mean"]) + 0.04, i,
            f"n={row['n']}  ({row['mean']:.2f})",
            va="center", ha="left", fontsize=8, color="#555555")

ax.set_yticks(y_pos)
ax.set_yticklabels([d.title() for d in forest_df["drug"]])
ax.set_xlabel("Mean user-level sentiment score (\u00b1 95% CI)")
ax.set_title("Forest Plot: Treatment Sentiment Among Long COVID Fatigue Users\n"
             "(Top 15 treatments by sample size)", pad=12)

ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.07),
    ncol=3,
    fontsize=9,
    frameon=False,
)

ax.set_xlim(-1.3, 1.5)
plt.tight_layout()
plt.show()

**What this chart shows:** Each blue dot is the average sentiment score for a treatment among fatigue users. The horizontal line through each dot is the 95% confidence interval -- the range where we believe the true average sentiment lies. The dashed grey line marks neutral (0.0), the green dotted line marks our positive threshold (0.7), and the red dotted line marks our negative threshold (-0.3).

**What this means:** Treatments whose entire CI sits to the right of 0 are consistently rated positively. Magnesium and vitamin D have both high mean scores and narrow-ish CIs, making them strong positive signals. LDN has the narrowest CI (thanks to n=79) centered solidly in positive territory. SSRIs are the only treatment whose CI extends below zero, confirming the mixed-to-negative pattern for this drug class among fatigue users. Wider CIs (smaller samples) signal more uncertainty -- the true effect could be anywhere within that range.

## 5. Question 3 -- Do Fatigue Users Respond Differently Than Non-Fatigue Users?

For each of the top treatments (by sample size among fatigue users), we compare user-level sentiment scores between the fatigue cohort and non-fatigue users using a **Mann-Whitney U test** (a non-parametric test appropriate for ordinal/skewed data). The rank-biserial correlation (r) measures effect size: positive r means fatigue users rate higher, negative r means non-fatigue users rate higher.

In [ ]:
# ---------- Build user-level scores for ALL users ----------
all_reports = pd.read_sql("""
    SELECT tr.report_id, tr.user_id, tr.drug_id, tr.sentiment,
           t.canonical_name AS drug
    FROM treatment_reports tr
    JOIN treatment t ON t.id = tr.drug_id
""", conn)

all_reports["score"] = all_reports["sentiment"].map(SENTIMENT_SCORES)

all_user_drug = (
    all_reports
    .groupby(["user_id", "drug"])
    .agg(avg_score=("score", "mean"))
    .reset_index()
)
all_user_drug["is_fatigue"] = all_user_drug["user_id"].isin(fatigue_user_ids)

# Top drugs from fatigue cohort (already filtered for generics)
top_drugs_for_comparison = (
    drug_summary
    .sort_values("n_users", ascending=False)
    .head(20)["drug"]
    .tolist()
)

comparison_results = []
for drug in top_drugs_for_comparison:
    fatigue_scores = all_user_drug.loc[
        (all_user_drug["drug"] == drug) & (all_user_drug["is_fatigue"]),
        "avg_score"
    ]
    non_fatigue_scores = all_user_drug.loc[
        (all_user_drug["drug"] == drug) & (~all_user_drug["is_fatigue"]),
        "avg_score"
    ]

    if len(fatigue_scores) < 5 or len(non_fatigue_scores) < 5:
        continue

    u_stat, p_val = stats.mannwhitneyu(
        fatigue_scores, non_fatigue_scores, alternative="two-sided"
    )
    # Rank-biserial correlation: r = 2*U/(n1*n2) - 1
    n1, n2 = len(fatigue_scores), len(non_fatigue_scores)
    r_effect = 2 * u_stat / (n1 * n2) - 1

    comparison_results.append({
        "Treatment": drug,
        "N fatigue": n1,
        "N non-fatigue": n2,
        "Mean (fatigue)": round(fatigue_scores.mean(), 3),
        "Mean (non-fatigue)": round(non_fatigue_scores.mean(), 3),
        "Diff": round(fatigue_scores.mean() - non_fatigue_scores.mean(), 3),
        "U statistic": u_stat,
        "p-value": p_val,
        "Effect r": round(r_effect, 3),
        "Significant": p_val < 0.05,
    })

comp_df = pd.DataFrame(comparison_results).sort_values("p-value")

# Format p-values for display
comp_display = comp_df.copy()
comp_display["p-value"] = comp_display["p-value"].apply(
    lambda x: f"{x:.3e}" if x < 0.001 else f"{x:.3f}"
)
display(comp_display.head(20))

**How to read this table:**

- **Diff** = mean sentiment among fatigue users minus mean sentiment among non-fatigue users. Positive means fatigue users rate the treatment higher.
- **Effect r** = rank-biserial correlation. Values near 0 mean no difference; +1 means fatigue users always rate higher; -1 means they always rate lower.
- **p-value** = Mann-Whitney U test. Small values mean the difference is unlikely due to chance alone.

**What this means:** Most treatments show no statistically significant difference between fatigue and non-fatigue users. This suggests that treatment experiences are broadly similar regardless of whether a user reports fatigue. The few treatments approaching significance (electrolytes, SSRIs) should be interpreted cautiously given the number of comparisons (risk of false positives from multiple testing).

In [ ]:
# ---------- Forest plot: fatigue vs non-fatigue mean difference ----------
# Show the difference in means with bootstrapped 95% CI
diff_forest = []
np.random.seed(42)

for _, row in comp_df.iterrows():
    drug = row["Treatment"]
    fatigue_scores = all_user_drug.loc[
        (all_user_drug["drug"] == drug) & (all_user_drug["is_fatigue"]),
        "avg_score"
    ].values
    non_fatigue_scores = all_user_drug.loc[
        (all_user_drug["drug"] == drug) & (~all_user_drug["is_fatigue"]),
        "avg_score"
    ].values

    observed_diff = fatigue_scores.mean() - non_fatigue_scores.mean()

    # Bootstrap 95% CI for the difference in means
    boot_diffs = []
    for _ in range(2000):
        boot_f = np.random.choice(fatigue_scores, size=len(fatigue_scores), replace=True)
        boot_nf = np.random.choice(non_fatigue_scores, size=len(non_fatigue_scores), replace=True)
        boot_diffs.append(boot_f.mean() - boot_nf.mean())

    ci_low = np.percentile(boot_diffs, 2.5)
    ci_high = np.percentile(boot_diffs, 97.5)

    diff_forest.append({
        "drug": drug,
        "diff": observed_diff,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "p": row["p-value"],
        "n_f": int(row["N fatigue"]),
        "n_nf": int(row["N non-fatigue"]),
    })

diff_df = pd.DataFrame(diff_forest).sort_values("diff", ascending=True)

fig, ax = plt.subplots(figsize=(9, 7))
y_pos = np.arange(len(diff_df))

# CI lines
ax.hlines(y_pos, diff_df["ci_low"], diff_df["ci_high"],
          color="#2c3e50", linewidth=1.5, zorder=2)

# Color dots by whether CI excludes 0
colors = ["#e67e22" if (row["ci_low"] > 0 or row["ci_high"] < 0) else "#3498db"
          for _, row in diff_df.iterrows()]
ax.scatter(diff_df["diff"], y_pos, c=colors, s=60, zorder=3,
           edgecolors="white", linewidth=0.5)

# Zero line
ax.axvline(0, color="grey", linewidth=0.8, linestyle="--", alpha=0.6,
           label="No difference")

# Labels
for i, (_, row) in enumerate(diff_df.iterrows()):
    label_text = f"{row['n_f']}v{row['n_nf']}  ({row['diff']:+.2f})"
    ax.text(max(row["ci_high"], row["diff"]) + 0.03, i,
            label_text, va="center", ha="left", fontsize=7.5, color="#555555")

ax.set_yticks(y_pos)
ax.set_yticklabels([d.title() for d in diff_df["drug"]])
ax.set_xlabel("Difference in mean sentiment (fatigue minus non-fatigue, 95% bootstrap CI)")
ax.set_title("Do Fatigue Users Rate Treatments Differently?\n"
             "(Orange dots: CI excludes zero)", pad=12)

ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.07),
    ncol=1,
    fontsize=9,
    frameon=False,
)

plt.tight_layout()
plt.show()

**What this chart shows:** Each dot represents the difference in average sentiment between fatigue users and non-fatigue users for a given treatment. Positive values (right of the dashed line) mean fatigue users rate the treatment more favorably; negative values mean the opposite. The horizontal lines are bootstrapped 95% confidence intervals. Orange dots indicate treatments where the CI excludes zero (a potentially meaningful difference); blue dots indicate no reliable difference.

**What this means:** For most treatments, the confidence interval crosses zero, meaning we cannot confidently say fatigue users respond differently. This is reassuring -- it suggests treatment sentiment data from the broader Long COVID community is likely applicable to fatigue sufferers as well. Where differences do appear, they are modest in magnitude and should be viewed with caution given the number of comparisons.

---

## 6. Tiered Recommendations

Based on user-reported sentiment from Long COVID communities, here is what the data says about reducing fatigue. Treatments are grouped by the strength of evidence in this dataset. **This is not medical advice** -- always discuss treatment changes with your healthcare provider.

In [ ]:
# ---------- Build tiered recommendations ----------
from IPython.display import Markdown, display as ipy_display

# Re-sort drug_summary by positive rate for recommendation ranking
rec_df = drug_summary.copy()

strong = []    # n >= 30 AND p < 0.05
moderate = []  # n >= 15 OR p < 0.10 (but not strong)
preliminary = []  # everything else

for _, row in rec_df.iterrows():
    drug_name = row["drug"].title()
    n = int(row["n_users"])
    pct = row["pct_positive"]
    p = row["binom_p"]
    mean_s = round(row["mean_score"], 2)

    if n >= 30 and p < 0.05:
        line = (f"**{drug_name}** -- {pct:.0f}% positive "
                f"(n={n}, p={p:.3f}). ")
        # Add plain-language note
        if drug_name == "Low Dose Naltrexone":
            line += ("Most-discussed treatment for fatigue with "
                     "consistently positive reports across the largest sample.")
        elif drug_name == "Antihistamines":
            line += ("Widely discussed, reflecting interest in mast cell "
                     "activation approaches. Individual responses vary.")
        elif drug_name == "Nicotine":
            line += ("Emerging community interest, especially via patches. "
                     "Positive but unconventional -- discuss with your doctor.")
        else:
            line += (f"Mean sentiment {mean_s} among fatigue users. "
                     "Worth discussing with your provider.")
        strong.append(line)

    elif n >= 15 or p < 0.10:
        line = (f"**{drug_name}** -- {pct:.0f}% positive "
                f"(n={n}, p={p:.3f}). ")
        line += (f"Mean sentiment {mean_s}. Sample size is moderate; "
                 "promising but needs more data.")
        moderate.append(line)

    else:
        line = (f"**{drug_name}** -- {pct:.0f}% positive "
                f"(n={n}). ")
        line += ("Too few reports to draw conclusions. "
                 "Listed here because it appeared in the dataset.")
        preliminary.append(line)

# Assemble markdown
md_parts = []

md_parts.append("### Strong evidence (n >= 30, p < 0.05)\n")
md_parts.append("These treatments have both a large enough sample and a "
                "statistically significant positive rate in our dataset.\n")
if strong:
    for s in strong:
        md_parts.append(f"- {s}")
else:
    md_parts.append("- No treatments met this threshold.")

md_parts.append("\n### Moderate evidence (n >= 15 or p < 0.10)\n")
md_parts.append("These treatments show promise but either have moderate sample "
                "sizes or borderline statistical significance.\n")
if moderate:
    for m in moderate:
        md_parts.append(f"- {m}")
else:
    md_parts.append("- No treatments in this tier.")

md_parts.append("\n### Preliminary (n < 15, not significant)\n")
md_parts.append("These treatments have too few reporters to draw reliable "
                "conclusions. They appear in the data but need much larger "
                "samples before we can make any recommendation.\n")
if preliminary:
    for p_item in preliminary[:15]:  # cap at 15 to avoid excessive length
        md_parts.append(f"- {p_item}")
    if len(preliminary) > 15:
        md_parts.append(f"- ... and {len(preliminary) - 15} more treatments "
                        "with fewer than 15 reporters.")
else:
    md_parts.append("- No treatments in this tier.")

ipy_display(Markdown("\n".join(md_parts)))

### Plain-language summary for patients

Based on what Long COVID patients are reporting online:

1. **Low dose naltrexone (LDN)** is the standout. It has the most discussion, the most reporters, and a consistently positive signal. If you are experiencing fatigue and have not tried LDN, it is the single strongest lead in this dataset.

2. **Supplements with good signals** include CoQ10, magnesium, vitamin D, creatine, and vitamin C. These are generally low-risk, widely available, and show positive sentiment among fatigue users. They are reasonable to discuss with your doctor.

3. **Antihistamines** (cetirizine, ketotifen, and others) are frequently discussed, especially by users with mast cell activation concerns. Results are more mixed than for LDN, suggesting they help some people substantially but not everyone.

4. **SSRIs** are the notable caution flag -- fatigue users report more mixed-to-negative sentiment compared to other treatments. This does not mean SSRIs are universally harmful, but it does suggest they may not be the best choice specifically for fatigue management.

5. **Nicotine patches** are an unconventional but increasingly discussed option with positive reports. This is early-stage community experimentation and warrants careful medical supervision.

**Remember:** This data comes from self-selected online reports, not clinical trials. What works for one person may not work for another. Always consult your healthcare provider before starting or changing treatments.

---

## 7. Reporting Bias Disclaimer

**Important limitations of this analysis:**

This analysis is based entirely on self-selected Reddit posts from Long COVID communities. Several forms of bias are present:

- **Selection bias:** Users who post about treatments are not representative of all patients. People with strong reactions (positive or negative) are more likely to post than those with neutral experiences.
- **Survivorship bias:** Users who stopped a treatment and left the community are invisible to this analysis.
- **Recall bias:** Users may remember and report dramatic improvements more readily than gradual or subtle changes.
- **Confounding:** Many users try multiple treatments simultaneously. We cannot isolate the effect of any single treatment.
- **Sentiment extraction:** Our NLP pipeline may misclassify some sentiments, particularly for nuanced or sarcastic posts.
- **Temporal snapshot:** This is a 1-month snapshot. Treatment trends and community sentiment shift over time.

The results reflect **reporting patterns in online communities**, not population-level treatment effects. This is not medical advice. Always consult a qualified healthcare provider before starting, stopping, or changing any treatment.

In [ ]:
conn.close()
print("Database connection closed.")